# add-sub-div-back-lambdas — worked example 1: Build a 4-entry BACK dict for add and div

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `add-sub-div-back-lambdas`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For elementwise ops the backward is a one-line lambda keyed by `(op_name, argnum)`. For `out = x + y` both partials are `1`, so each lambda just returns the incoming grad `g`. For `out = x / y` the arg0 partial is `1/y` and the arg1 partial is `-x/y**2`. Storing them in a dict mirrors how an autograd dispatcher looks back-fns up at reverse time.

## Worked solution

**Goal.** Build `BACK`, a dict mapping `(op_name, argnum)` to a backward lambda, for just `add` and `div` (four entries). Each lambda has signature `(g, o, x, y) -> Tensor`.

**Step 1 — write the forward and differentiate.** For `out = x + y`, `d(out)/dx = 1` and `d(out)/dy = 1`. By the chain rule the gradient flowing back to each input is `g * 1 = g`. So both add lambdas return `g` unchanged. This is *why* additions are gradient-routers: they copy the upstream gradient to every input.

**Step 2 — differentiate div.** For `out = x / y`, treat `y` as constant to get `d(out)/dx = 1/y`. So `BACK[('div', 0)] = lambda g, o, x, y: g / y`. Treating `x` as constant, `out = x * y**(-1)`, so `d(out)/dy = -x * y**(-2) = -x/y**2`. Hence `BACK[('div', 1)] = lambda g, o, x, y: -g * x / (y * y)`.

**Step 3 — key by tuple.** We use `(op_name, argnum)` tuples as keys so the dispatcher can index `BACK[(name, arg)]` without any `if/elif` chain. The `o` (output) slot is unused for add/div but kept in the signature so every lambda is interchangeable.

**Step 4 — exercise it.** We pick concrete tensors and a synthetic upstream grad, then call the lambdas and compare against PyTorch autograd to confirm correctness.

In [ ]:
BACK = {
    ('add', 0): lambda g, o, x, y: g,
    ('add', 1): lambda g, o, x, y: g,
    ('div', 0): lambda g, o, x, y: g / y,
    ('div', 1): lambda g, o, x, y: -g * x / (y * y),
}

t.manual_seed(0)
x = t.randn(4, requires_grad=True)
y = (t.randn(4).abs() + 0.5).requires_grad_(True)  # leaf, denom away from 0
out = x / y
g = t.randn(4)

# manual backward via BACK
dx = BACK[('div', 0)](g, out, x.detach(), y.detach())
dy = BACK[('div', 1)](g, out, x.detach(), y.detach())

# autograd reference
out.backward(g)
print('dx match:', t.allclose(dx, x.grad, atol=1e-5))
print('dy match:', t.allclose(dy, y.grad, atol=1e-5))